In [ ]:
print("--- Installing and Importing Libraries ---")
!pip install -U transformers datasets accelerate evaluate scikit-learn --quiet

import os
import pandas as pd
import numpy as np
import torch
from PIL import Image as PILImage
from datasets import Dataset, DatasetDict
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

from torchvision.transforms import (
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomResizedCrop,
    RandomRotation,
    ColorJitter,
    ToTensor,
    Resize,
    CenterCrop,
)

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
    login(token=hf_token)
    print("Hugging Face login successful.")
except Exception:
    print("Hugging Face token not found. Model will not be pushed to the Hub.")

print("\n--- Starting Data Loading and Preprocessing ---")

# Step 1: Load the metadata
metadata_path = "/kaggle/input/skin-cancer/metadata.csv"
df = pd.read_csv(metadata_path)

# Step 2: Create a dictionary of all image file paths
all_image_paths = {}
path_dirs = [
    '/kaggle/input/skin-cancer/imgs_part_1/imgs_part_1/',
    '/kaggle/input/skin-cancer/imgs_part_2/imgs_part_2/',
    '/kaggle/input/skin-cancer/imgs_part_3/imgs_part_3/'
]
for path_dir in path_dirs:
    if os.path.exists(path_dir):
        for fname in os.listdir(path_dir):
            all_image_paths[fname] = os.path.join(path_dir, fname)

# Step 3: Prepare the final DataFrame
df['path'] = df['img_id'].map(all_image_paths.get)
df['label'] = df['diagnostic']
df_clean = df[['path', 'label']].dropna().copy()
print(f"Found {len(df_clean)} matching image files and labels.")

# Step 4: Create the Hugging Face Dataset from pandas
raw_dataset = Dataset.from_pandas(df_clean)

# Step 5: Encode string labels into integers
raw_dataset = raw_dataset.class_encode_column("label")

# Step 6: Process all images into pixel values (upfront processing)
model_name = "Anwarkh1/Skin_Cancer-Image_Classification"
processor = ViTImageProcessor.from_pretrained(model_name)

def preprocess_images(examples):
    images = [PILImage.open(path).convert("RGB") for path in examples["path"]]
    processed_inputs = processor(images, return_tensors="pt")
    examples['pixel_values'] = processed_inputs['pixel_values']
    return examples

processed_dataset = raw_dataset.map(preprocess_images, batched=True, batch_size=100)

processed_dataset = processed_dataset.remove_columns(['path'])

# Step 7: Create Train/Test Splits
train_test_split_dataset = processed_dataset.train_test_split(test_size=0.2, stratify_by_column="label")
final_datasets = DatasetDict({
    'train': train_test_split_dataset['train'],
    'test': train_test_split_dataset['test']
})

final_datasets.set_format('torch')

print("\n--- Preprocessing Complete. Final Dataset Info: ---")
print(final_datasets)
print("\nClass Names:", final_datasets['train'].features['label'].names)



print("\n--- Setting up for Training ---")

# Step 1: Define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    acc = accuracy_score(labels, predictions)
    bacc = balanced_accuracy_score(labels, predictions)
    return {'accuracy': acc, 'balanced_accuracy': bacc, 'f1': f1, 'precision': precision, 'recall': recall}

# Step 2: Load the model and configure it
labels = final_datasets['train'].features['label'].names
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in id2label.items()}

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
print("Model loaded and configured for new labels.")

# Step 3: Calculate class weights on the CPU
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(np.array(train_test_split_dataset['train']['label'])),
    y=np.array(train_test_split_dataset['train']['label'])
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print(f"Calculated Class Weights: {class_weights}")

# Step 4: Create a custom Trainer to use the weighted loss
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = self.class_weights.to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Step 5: Define training arguments
hub_model_id = "bnmbanhmi/seekwell_skincancer_v2" 

training_args = TrainingArguments(
  output_dir="/kaggle/working/seekwell_model_v6_improved",
  num_train_epochs=15,
  learning_rate=2e-5, 
  per_device_train_batch_size=32,
  weight_decay=0.01,
  eval_strategy="epoch",
  save_strategy="epoch",
  fp16=True,
  load_best_model_at_end=True,
  metric_for_best_model="balanced_accuracy",
  push_to_hub=True,
  hub_model_id=hub_model_id,
  hub_strategy="every_save",
  report_to="none",
  logging_strategy="epoch", 
  lr_scheduler_type="cosine", 
  warmup_ratio=0.1, 
)

# Step 6: Instantiate the CustomTrainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["test"],
)
trainer.class_weights = class_weights_tensor


print("\n--- Starting Training ---")
train_results = trainer.train()

print("\n--- Final Evaluation on Test Set ---")
metrics = trainer.evaluate()
print("\n--- Final Performance Metrics (from best model) ---")
print(metrics)

trainer.save_model()
trainer.save_state()
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)
print(f"\n--- Model saved to {training_args.output_dir} ---")


In [ ]:
# ==========================================
# ZENSTAR024 - SKIN CANCER DETECTION
# Exploratory Data Analysis (EDA)
# ==========================================

import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from PIL import Image
from collections import Counter
from IPython.display import display

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="viridis")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print("Libraries imported successfully!")
print("Project: ZENSTAR024 Skin Cancer Analysis")

In [ ]:
# ==========================================
# DATASET LOADING
# ==========================================

BASE_PATH = "/kaggle/input"

csv_files = glob.glob(
    BASE_PATH + "/**/HAM10000_metadata.csv",
    recursive=True
)

if not csv_files:
    csv_files = glob.glob(
        BASE_PATH + "/**/*.csv",
        recursive=True
    )

if not csv_files:
    raise FileNotFoundError(
        "CSV file not found. Check Kaggle Input dataset."
    )

metadata_path = csv_files[0]

df = pd.read_csv(metadata_path)

print("Dataset loaded successfully!")
print("Metadata Path:", metadata_path)

display(df.head())

In [ ]:
# ==========================================
# DATASET OVERVIEW
# ==========================================

print("Dataset Shape:", df.shape)

print("\nNumber of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
display(df.describe(include="all").T)

In [ ]:
# ==========================================
# MISSING VALUES
# ==========================================

missing = df.isnull().sum()

missing_df = pd.DataFrame({
    "Column": missing.index,
    "Missing Values": missing.values,
    "Missing Percentage": (
        missing.values / len(df) * 100
    ).round(2)
})

display(missing_df)

plt.figure(figsize=(12, 5))

sns.heatmap(
    df.isnull(),
    cbar=False,
    cmap="magma",
    yticklabels=False
)

plt.title("Missing Values Heatmap")
plt.xlabel("Dataset Columns")
plt.show()

In [ ]:
# ==========================================
# DUPLICATE ANALYSIS
# ==========================================

duplicates = df.duplicated().sum()

print("Total Duplicate Rows:", duplicates)

print("Unique Image IDs:", df["image_id"].nunique()
      if "image_id" in df.columns else "Column unavailable")

print("Total Records:", len(df))

In [ ]:
# ==========================================
# SKIN CANCER DIAGNOSIS DISTRIBUTION
# ==========================================

if "dx" not in df.columns:
    raise ValueError("Diagnosis column 'dx' not found.")

diagnosis_counts = df["dx"].value_counts()

display(
    diagnosis_counts.rename("Image Count").to_frame()
)

plt.figure(figsize=(13, 6))

sns.barplot(
    x=diagnosis_counts.index,
    y=diagnosis_counts.values
)

plt.title("Skin Lesion Diagnosis Distribution")
plt.xlabel("Diagnosis Category")
plt.ylabel("Number of Images")

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# DIAGNOSIS PERCENTAGE DISTRIBUTION
# ==========================================

plt.figure(figsize=(9, 9))

plt.pie(
    diagnosis_counts.values,
    labels=diagnosis_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.8
)

plt.title("Skin Lesion Category Percentage")
plt.show()

In [ ]:
# ==========================================
# PATIENT AGE ANALYSIS
# ==========================================

if "age" in df.columns:

    df["age"] = pd.to_numeric(df["age"], errors="coerce")

    plt.figure(figsize=(12, 6))

    sns.histplot(
        df["age"].dropna(),
        bins=25,
        kde=True
    )

    plt.title("Patient Age Distribution")
    plt.xlabel("Age")
    plt.ylabel("Frequency")

    plt.show()

    print("Average Age:", round(df["age"].mean(), 2))
    print("Median Age:", round(df["age"].median(), 2))

else:
    print("Age column not available.")

In [ ]:
# ==========================================
# GENDER DISTRIBUTION
# ==========================================

if "sex" in df.columns:

    gender_counts = df["sex"].value_counts()

    plt.figure(figsize=(8, 5))

    sns.barplot(
        x=gender_counts.index,
        y=gender_counts.values
    )

    plt.title("Patient Gender Distribution")
    plt.xlabel("Gender")
    plt.ylabel("Number of Records")

    plt.show()

    display(gender_counts.to_frame("Count"))

else:
    print("Gender column not available.")

In [ ]:
# ==========================================
# BODY LOCATION ANALYSIS
# ==========================================

if "localization" in df.columns:

    location_counts = (
        df["localization"]
        .value_counts()
        .head(15)
    )

    plt.figure(figsize=(14, 6))

    sns.barplot(
        x=location_counts.values,
        y=location_counts.index
    )

    plt.title("Top 15 Skin Lesion Locations")
    plt.xlabel("Number of Images")
    plt.ylabel("Body Location")

    plt.tight_layout()
    plt.show()

else:
    print("Localization column not available.")

In [ ]:
# ==========================================
# IMAGE DIRECTORY DETECTION
# ==========================================

image_extensions = ["*.jpg", "*.jpeg", "*.png"]

image_paths = []

for extension in image_extensions:
    image_paths.extend(
        glob.glob(
            BASE_PATH + "/**/" + extension,
            recursive=True
        )
    )

image_map = {}

for path in image_paths:
    image_id = os.path.splitext(
        os.path.basename(path)
    )[0]

    image_map[image_id] = path

print("Total Images Found:", len(image_map))

if len(image_map) == 0:
    raise FileNotFoundError(
        "No images found. Attach the skin cancer image dataset."
    )

df["image_path"] = df["image_id"].map(image_map)

print("Matched Images:", df["image_path"].notna().sum())
print("Unmatched Images:", df["image_path"].isna().sum())

In [ ]:
# ==========================================
# SKIN LESION IMAGE GALLERY
# ==========================================

sample_df = (
    df.dropna(subset=["image_path", "dx"])
    .groupby("dx", group_keys=False)
    .head(2)
)

fig, axes = plt.subplots(
    2, 7,
    figsize=(20, 7)
)

axes = axes.flatten()

for ax in axes:
    ax.axis("off")

for ax, (_, row) in zip(axes, sample_df.iterrows()):

    img = Image.open(row["image_path"]).convert("RGB")

    ax.imshow(img)
    ax.set_title(
        f"Diagnosis: {row['dx']}",
        fontsize=10
    )

    ax.axis("off")

plt.suptitle(
    "Actual Skin Lesion Image Gallery",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# SKIN LESION MEDICAL INFORMATION
# EDUCATIONAL REFERENCE ONLY
# ==========================================

disease_info = {
    "mel": {
        "Name": "Melanoma",
        "Signs": "Change in size, shape or colour; irregular borders.",
        "Note": "A potentially serious skin cancer requiring medical assessment."
    },

    "nv": {
        "Name": "Melanocytic Nevi",
        "Signs": "Usually stable, round or oval pigmented moles.",
        "Note": "Generally benign, but changing lesions need assessment."
    },

    "bcc": {
        "Name": "Basal Cell Carcinoma",
        "Signs": "Pearly bump, non-healing sore or bleeding area.",
        "Note": "A common type of skin cancer."
    },

    "akiec": {
        "Name": "Actinic Keratoses",
        "Signs": "Rough, scaly or crusty patches.",
        "Note": "Some lesions can progress to squamous cell carcinoma."
    },

    "bkl": {
        "Name": "Benign Keratosis",
        "Signs": "Waxy, rough or stuck-on appearing growth.",
        "Note": "Usually non-cancerous."
    },

    "df": {
        "Name": "Dermatofibroma",
        "Signs": "Small, firm skin-coloured or brown bump.",
        "Note": "Usually benign."
    },

    "vasc": {
        "Name": "Vascular Lesions",
        "Signs": "Red, purple or bluish skin marks.",
        "Note": "Many are benign; clinical examination may be needed."
    }
}

for code, info in disease_info.items():

    print("\n" + "=" * 55)
    print("Category:", code)
    print("Disease:", info["Name"])
    print("Common Signs:", info["Signs"])
    print("Information:", info["Note"])

In [ ]:
# ==========================================
# EDA SUMMARY
# ==========================================

print("=" * 60)
print("ZENSTAR024 - SKIN CANCER DATASET SUMMARY")
print("=" * 60)

print("Total Records:", len(df))

print(
    "Total Diagnosis Categories:",
    df["dx"].nunique()
)

print(
    "Total Available Images:",
    len(image_map)
)

print(
    "Matched Image Records:",
    df["image_path"].notna().sum()
)

print(
    "Missing Values:",
    df.isnull().sum().sum()
)

print(
    "Duplicate Rows:",
    df.duplicated().sum()
)

print("\nDiagnosis Counts:")
display(df["dx"].value_counts().to_frame("Count"))

print("\nEDA Completed Successfully!")